New Notebook Created by Jupyter MCP Server

In [1]:
# Panel B — mne.viz.Brain frontal view of the 23 channel layout on fsaverage.
# Neutral color (orange) for all channels; no XAI overlay (per user choice).
import os
os.environ["PYVISTA_OFF_SCREEN"] = "true"   # headless if needed
os.environ["MPLBACKEND"] = "Agg"

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import mne

REPO = Path('/home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method')
ELC = REPO / 'data' / 'brainproducts-RNP-BA-128-custom.elc'
GNG_DIR = REPO / 'data' / 'raw' / 'anxiety' / 'EA062' / 'GNG'
OUT_DIR = REPO / 'research' / 'paper-materials' / 'figures' / '_intermediate'
OUT_DIR.mkdir(parents=True, exist_ok=True)

mne.set_log_level("ERROR")
print("MNE version:", mne.__version__)
print("Output dir:", OUT_DIR)
print("Source NIRx dir exists:", GNG_DIR.exists())

MNE version: 1.12.0
Output dir: /home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method/research/paper-materials/figures/_intermediate
Source NIRx dir exists: True


In [2]:
# Build the 23-channel HBO info using the same recipe as notebook 2_:
#   read_raw_nirx → set_montage → optical_density → beer_lambert → pick(hbo) → info
raw = mne.io.read_raw_nirx(str(GNG_DIR), preload=False, verbose=False)
montage = mne.channels.read_custom_montage(str(ELC), head_size=0.0825)
raw.set_montage(montage)

raw_od = mne.preprocessing.nirs.optical_density(raw, verbose=False)
raw_haemo = mne.preprocessing.nirs.beer_lambert_law(raw_od, ppf=6.0)
hbo = raw_haemo.copy().pick_types(fnirs='hbo', verbose=False)
info = hbo.info
print("n_channels:", len(info.ch_names))
print("first 3 channels:", info.ch_names[:3])
print("first 3 channels loc[0:3] (chan_pos head-CTF):")
for ch in info['chs'][:3]:
    print(f"  {ch['ch_name']}: chan_pos={ch['loc'][:3]}, src={ch['loc'][3:6]}, det={ch['loc'][6:9]}")

n_channels: 23
first 3 channels: ['S1_D1 hbo', 'S1_D3 hbo', 'S2_D2 hbo']
first 3 channels loc[0:3] (chan_pos head-CTF):
  S1_D1 hbo: chan_pos=[-0.02004749  0.10087929  0.04232577], src=[-0.02745245  0.09903983  0.03103963], det=[-0.01264252  0.10271874  0.05361191]
  S1_D3 hbo: chan_pos=[-0.03765144  0.09152755  0.0460646 ], src=[-0.02745245  0.09903983  0.03103963], det=[-0.04785044  0.08401526  0.06108958]
  S2_D2 hbo: chan_pos=[0.01957626 0.10039223 0.04243965], src=[0.01231624 0.10239345 0.05357777], det=[0.02683627 0.09839101 0.03130153]


/tmp/ipykernel_1771194/2635691064.py:3: RuntimeWarning: MNE has not been tested with Aurora version 2021.9.0-20-g15526401-dirty
  raw = mne.io.read_raw_nirx(str(GNG_DIR), preload=False, verbose=False)


In [3]:
# Fetch fsaverage (cached after first run) and prep colours.
fs_root = mne.datasets.fetch_fsaverage(verbose=False)
subjects_dir = Path(fs_root).parent
print("fsaverage at:", fs_root)

n_ch = len(info.ch_names)
# Neutral orange for all channels (no XAI overlay).
fnirs_colors = np.tile(np.array([1.0, 0.647, 0.0, 1.0]), (n_ch, 1))  # RGBA
print("colour array shape:", fnirs_colors.shape)

fsaverage at: /home/user/mne_data/MNE-fsaverage-data/fsaverage
colour array shape: (23, 4)


In [4]:
# Render mne.viz.Brain — frontal (rostral) view, neutral colour for all channels.
# Notebook 2_ pattern. Set 3-D backend to PyVista (off-screen).
mne.viz.set_3d_backend("pyvistaqt")

brain = mne.viz.Brain(
    "fsaverage",
    subjects_dir=str(subjects_dir),
    background="white",
    cortex="low_contrast",
    alpha=0.3,
    units='mm',
    size=(1200, 1000),
)

brain.add_sensors(
    info,
    trans="fsaverage",
    fnirs=['channels'],
    sensor_scales=4.5,
    sensor_colors={'fnirs': fnirs_colors[:, :3]},  # RGB only for add_sensors
)

# Frontal view (looking at the front of the head)
brain.show_view(view='rostral', elevation=70)

panel_b_path = OUT_DIR / 'panel_B_brain.png'
brain.save_image(str(panel_b_path), mode='rgba')
print("Saved:", panel_b_path)
print("File size:", panel_b_path.stat().st_size, "bytes")

QtBindingsNotFoundError: No Qt bindings could be found

In [5]:
# Use the 'notebook' 3-D backend (no Qt required).
try:
    mne.viz.set_3d_backend("notebook")
    print("Using 3D backend:", mne.viz.get_3d_backend())
except Exception as e:
    print("notebook backend failed:", e)
    # Try plain pyvista off-screen
    import pyvista as pv
    pv.OFF_SCREEN = True
    pv.start_xvfb() if hasattr(pv, "start_xvfb") else None
    print("Falling back to pyvista off-screen mode")

Using 3D backend: notebook


In [6]:
# Render mne.viz.Brain with the notebook backend.
brain = mne.viz.Brain(
    "fsaverage",
    subjects_dir=str(subjects_dir),
    background="white",
    cortex="low_contrast",
    alpha=0.3,
    units='mm',
    size=(1200, 1000),
)

brain.add_sensors(
    info,
    trans="fsaverage",
    fnirs=['channels'],
    sensor_scales=4.5,
    sensor_colors={'fnirs': fnirs_colors[:, :3]},
)

brain.show_view(view='rostral', elevation=70)

panel_b_path = OUT_DIR / 'panel_B_brain.png'
brain.save_image(str(panel_b_path), mode='rgba')
print("Saved:", panel_b_path)
print("File size:", panel_b_path.stat().st_size, "bytes")

2026-05-12 03:15:58.510 (   7.466s) [    7D14BA4DF740]vtkXOpenGLRenderWindow.:1460  WARN| bad X server connection. DISPLAY=


Saved: /home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method/research/paper-materials/figures/_intermediate/panel_B_brain.png
File size: 346434 bytes


In [7]:
# Iteration 2: explicit frontal view via azimuth/elevation, brighter cortex,
# bigger sensor markers, larger output canvas.
brain.close()

brain = mne.viz.Brain(
    "fsaverage",
    subjects_dir=str(subjects_dir),
    background="white",
    cortex="classic",         # solid grey, no overlay
    alpha=0.55,               # more visible than 0.3
    units='mm',
    size=(1400, 1100),
)

brain.add_sensors(
    info,
    trans="fsaverage",
    fnirs=['channels'],
    sensor_scales=6.0,
    sensor_colors={'fnirs': fnirs_colors[:, :3]},
)

# True frontal view: look at the rostral pole from in front, slightly tilted.
brain.show_view(azimuth=90, elevation=80, distance=420)

panel_b_path = OUT_DIR / 'panel_B_brain.png'
brain.save_image(str(panel_b_path), mode='rgba')
print("Saved:", panel_b_path)
import imageio.v3 as iio
img = iio.imread(str(panel_b_path))
print("Image shape:", img.shape)

Saved: /home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method/research/paper-materials/figures/_intermediate/panel_B_brain.png
Image shape: (1100, 1400, 4)


In [8]:
# Iteration 3: try multiple view presets / explicit azimuth, save each as a
# preview so we can pick the best one.
views_to_try = [
    ('frontal',  dict(view='frontal')),
    ('rostral',  dict(view='rostral')),
    ('azim180_el0',   dict(azimuth=180, elevation=0)),
    ('azim270_el20',  dict(azimuth=270, elevation=20)),
    ('azim180_el40',  dict(azimuth=180, elevation=40)),
]

previews = []
for name, kwargs in views_to_try:
    brain.show_view(**kwargs)
    out = OUT_DIR / f'panel_B_preview_{name}.png'
    brain.save_image(str(out), mode='rgba')
    previews.append(out)
    print(f"  {name:18s} → {out.name}")
print("done.")

  frontal            → panel_B_preview_frontal.png
  rostral            → panel_B_preview_rostral.png


  azim180_el0        → panel_B_preview_azim180_el0.png


  azim270_el20       → panel_B_preview_azim270_el20.png


  azim180_el40       → panel_B_preview_azim180_el40.png
done.


In [9]:
# Final Panel B with view='rostral' (no elevation override).
brain.show_view(view='rostral')

panel_b_path = OUT_DIR / 'panel_B_brain.png'
brain.save_image(str(panel_b_path), mode='rgba')
print("Final Panel B saved:", panel_b_path)
img = iio.imread(str(panel_b_path))
print("shape:", img.shape, "size_kb:", panel_b_path.stat().st_size // 1024)

# Clean up the preview files now that we've chosen.
for p in OUT_DIR.glob('panel_B_preview_*.png'):
    p.unlink()
print("cleaned up preview files.")

Final Panel B saved: /home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method/research/paper-materials/figures/_intermediate/panel_B_brain.png
shape: (1100, 1400, 4) size_kb: 423
cleaned up preview files.


In [10]:
# Crop Panel B's PNG to remove whitespace around the brain.
import numpy as np
import imageio.v3 as iio

panel_b_path = OUT_DIR / 'panel_B_brain.png'
img = iio.imread(str(panel_b_path))
print("Original:", img.shape)

# Find the bounding box of non-white pixels (RGBA: white background).
# White = (R~255, G~255, B~255). Use brightness threshold.
rgb = img[:, :, :3].astype(np.int32)
is_content = (rgb.sum(axis=2) < 740)  # not near-white
ys, xs = np.where(is_content)
if len(ys) == 0:
    print("WARNING: no content found")
else:
    y0, y1 = ys.min(), ys.max()
    x0, x1 = xs.min(), xs.max()
    pad = 40
    y0 = max(0, y0 - pad); y1 = min(img.shape[0], y1 + pad)
    x0 = max(0, x0 - pad); x1 = min(img.shape[1], x1 + pad)
    cropped = img[y0:y1, x0:x1]
    print("Cropped:", cropped.shape, "from bbox", (y0, y1, x0, x1))
    iio.imwrite(str(panel_b_path), cropped)
    print("Saved cropped Panel B:", panel_b_path)

Original: (1100, 1400, 4)
Cropped: (880, 829, 4) from bbox (np.int64(82), np.int64(962), np.int64(284), np.int64(1113))
Saved cropped Panel B: /home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method/research/paper-materials/figures/_intermediate/panel_B_brain.png


In [11]:
# Re-render the brain panel with notebook 2_ recipe:
#   - view='rostral', elevation=70 (default notebook 2_ angle)
#   - ROI coloring (VMPFC blue, DMPFC green, DLPFC red, default orange)
#   - channel-number labels (1..23) via PyVista add_point_labels
brain.close()

# Channel-name ROI mapping copied verbatim from notebook 2_'s vmpfc/dmpfc/dlpfc lists.
vmpfc_idx = sorted({0, 3, 2})
dmpfc_idx = sorted({7, 17, 18, 9, 19, 10, 11, 21})
dlpfc_idx = sorted({6, 8, 16, 15, 13, 14, 22, 20})

n_ch = len(info.ch_names)
# Default: orange (matches notebook 2_)
fnirs_colors = np.tile(np.array([1.0, 0.647, 0.0]), (n_ch, 1))
blue, green, red = np.array([0.0, 0.0, 1.0]), np.array([0.0, 1.0, 0.0]), np.array([1.0, 0.0, 0.0])
for i in vmpfc_idx:  fnirs_colors[i] = blue
for i in dmpfc_idx:  fnirs_colors[i] = green
for i in dlpfc_idx:  fnirs_colors[i] = red

print("ROI counts:")
print(f"  VMPFC (blue):    {len(vmpfc_idx):2d}  channels: {[info.ch_names[i] for i in vmpfc_idx]}")
print(f"  DMPFC (green):   {len(dmpfc_idx):2d}  channels: {[info.ch_names[i] for i in dmpfc_idx]}")
print(f"  DLPFC (red):     {len(dlpfc_idx):2d}  channels: {[info.ch_names[i] for i in dlpfc_idx]}")
default_idx = [i for i in range(n_ch) if i not in set(vmpfc_idx + dmpfc_idx + dlpfc_idx)]
print(f"  default (orange):{len(default_idx):2d}  channels: {[info.ch_names[i] for i in default_idx]}")

ROI counts:
  VMPFC (blue):     3  channels: ['S1_D1 hbo', 'S2_D2 hbo', 'S2_D1 hbo']
  DMPFC (green):    8  channels: ['S3_D4 hbo', 'S4_D4 hbo', 'S4_D5 hbo', 'S4_D7 hbo', 'S7_D4 hbo', 'S7_D6 hbo', 'S7_D7 hbo', 'S8_D7 hbo']
  DLPFC (red):      8  channels: ['S3_D3 hbo', 'S3_D6 hbo', 'S5_D5 hbo', 'S5_D8 hbo', 'S6_D3 hbo', 'S6_D6 hbo', 'S8_D5 hbo', 'S8_D8 hbo']
  default (orange): 4  channels: ['S1_D3 hbo', 'S2_D5 hbo', 'S3_D1 hbo', 'S5_D2 hbo']


In [12]:
# Compute sensor positions in MRI/fsaverage mm space for channel-number labels.
# add_sensors(..., trans='fsaverage') applies the fsaverage trans (head -> MRI).
fs_trans = mne.coreg.coregister_fiducials(info, "fiducials", subjects_dir=str(subjects_dir)) if False else None
# Easier: load the standard fsaverage head->MRI transform.
from mne.transforms import _ensure_trans, read_trans
import mne

# fsaverage trans bundled with MNE
fs_trans_path = subjects_dir / "fsaverage" / "bem" / "fsaverage-trans.fif"
print("Standard fsaverage trans exists:", fs_trans_path.exists())

# When add_sensors is called with trans='fsaverage', MNE internally loads
# fsaverage-trans.fif. We mirror that for label coords.
trans = mne.read_trans(str(fs_trans_path))
print("trans:", trans)

# Channel head positions (loc[0:3]) are in HEAD coords, in metres.
head_positions = np.array([ch["loc"][:3] for ch in info["chs"]])
mri_positions_m = mne.transforms.apply_trans(trans["trans"], head_positions)
mri_positions_mm = mri_positions_m * 1000.0
print("Head positions (sample):", head_positions[0])
print("MRI positions mm (sample):", mri_positions_mm[0])

Standard fsaverage trans exists: True
trans: <Transform | head->MRI (surface RAS)>
[[ 0.9999938  -0.00355761 -0.00000057  0.00187336]
 [ 0.00355187  0.99838907 -0.05662654 -0.02879576]
 [ 0.00020204  0.05662593  0.99839538 -0.0412941 ]
 [ 0.          0.          0.          1.        ]]
Head positions (sample): [-0.02004749  0.10087929  0.04232577]
MRI positions mm (sample): [-18.53291117  69.45304574   6.67208229]


In [13]:
# Render the final brain figure: rostral view + elevation=70 (notebook 2_ default)
# + ROI coloring + channel-number labels.

brain = mne.viz.Brain(
    "fsaverage",
    subjects_dir=str(subjects_dir),
    background="white",
    cortex="low_contrast",
    alpha=0.3,                       # transparent (matches notebook 2_)
    units='mm',
    size=(1400, 1200),
)

brain.add_sensors(
    info,
    trans="fsaverage",
    fnirs=['channels'],
    sensor_scales=4.5,
    sensor_colors={'fnirs': fnirs_colors},
)

# Add channel-number labels (1..23) above each sensor.
# Use add_point_labels — labels float in 3D, follow the camera.
labels = [str(i + 1) for i in range(n_ch)]
# Offset labels slightly above the sensor to avoid overlap with the dot.
label_offset = np.zeros_like(mri_positions_mm)
label_offset[:, 2] = 6.0   # +6 mm in z (superior)
brain.plotter.add_point_labels(
    mri_positions_mm + label_offset,
    labels,
    font_size=18,
    text_color="black",
    point_color="white",
    point_size=1,            # invisible points (we just want labels)
    show_points=False,
    always_visible=True,
    shape=None,
    margin=0,
    name="ch_numbers",
)

brain.show_view(view='rostral', elevation=70)

fig_brain_path = REPO / 'research' / 'paper-materials' / 'figures' / 'fig_montage_brain.png'
brain.save_image(str(fig_brain_path), mode='rgba')
print("Saved:", fig_brain_path)
img_b = iio.imread(str(fig_brain_path))
print("shape:", img_b.shape)

Saved: /home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method/research/paper-materials/figures/fig_montage_brain.png
shape: (1200, 1400, 4)


In [14]:
# Crop fig_montage_brain.png to remove the whitespace; also write SVG sibling
# (by encoding the same PNG bytes as a base64-embedded SVG, which is the
# cleanest "vector wrapper" we can do for a raster screenshot).
img = iio.imread(str(fig_brain_path))
print("Original:", img.shape)

rgb = img[:, :, :3].astype(np.int32)
is_content = (rgb.sum(axis=2) < 740)
ys, xs = np.where(is_content)
y0, y1 = ys.min(), ys.max()
x0, x1 = xs.min(), xs.max()
pad = 50
y0 = max(0, y0 - pad); y1 = min(img.shape[0], y1 + pad)
x0 = max(0, x0 - pad); x1 = min(img.shape[1], x1 + pad)
cropped = img[y0:y1, x0:x1]
print("Cropped:", cropped.shape)
iio.imwrite(str(fig_brain_path), cropped)

# SVG wrapper (embeds the cropped PNG as base64) for paper-materials parity
import base64
svg_path = fig_brain_path.with_suffix(".svg")
with open(fig_brain_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("ascii")
H, W = cropped.shape[:2]
svg = (
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" '
    f'viewBox="0 0 {W} {H}">'
    f'<image href="data:image/png;base64,{b64}" width="{W}" height="{H}"/>'
    f'</svg>'
)
svg_path.write_text(svg)
print("Cropped PNG + SVG written:", fig_brain_path.name, svg_path.name)
print("Cropped size kb:", fig_brain_path.stat().st_size // 1024)

Original: (1200, 1400, 4)
Cropped: (957, 936, 4)
Cropped PNG + SVG written: fig_montage_brain.png fig_montage_brain.svg
Cropped size kb: 428


In [15]:
# Brain orientation: try alternative views so the rostral pole (front of head)
# ends up at the TOP of the image — matching the optode-layout figure.
brain.close()

brain = mne.viz.Brain(
    "fsaverage", subjects_dir=str(subjects_dir), background="white",
    cortex="low_contrast", alpha=0.3, units='mm', size=(1400, 1200),
)
brain.add_sensors(
    info, trans="fsaverage", fnirs=['channels'],
    sensor_scales=4.5,
    sensor_colors={'fnirs': fnirs_colors},
)
brain.plotter.add_point_labels(
    mri_positions_mm + label_offset,
    labels,
    font_size=20,
    text_color="black",
    point_color="white",
    point_size=1,
    show_points=False,
    always_visible=True,
    shape="rounded_rect",
    shape_color="white",
    shape_opacity=0.85,
    margin=2,
    name="ch_numbers",
)

views_to_try = [
    ('caudal_el70',     dict(view='caudal',  elevation=70)),
    ('rostral_el-70',   dict(view='rostral', elevation=-70)),
    ('rostral_el70_roll180', dict(view='rostral', elevation=70, roll=180)),
    ('dorsal',          dict(view='dorsal')),
]
preview_paths = []
for name, kwargs in views_to_try:
    brain.show_view(**kwargs)
    p = OUT_DIR / f'brain_preview_{name}.png'
    brain.save_image(str(p), mode='rgba')
    preview_paths.append((name, p))
    print(f"  {name:30s} → {p.name}")

  caudal_el70                    → brain_preview_caudal_el70.png
  rostral_el-70                  → brain_preview_rostral_el-70.png


  rostral_el70_roll180           → brain_preview_rostral_el70_roll180.png


  dorsal                         → brain_preview_dorsal.png


In [16]:
# Try variations on dorsal view + tighter zoom on the prefrontal region.
brain.close()
brain = mne.viz.Brain(
    "fsaverage", subjects_dir=str(subjects_dir), background="white",
    cortex="low_contrast", alpha=0.3, units='mm', size=(1400, 1200),
)
brain.add_sensors(
    info, trans="fsaverage", fnirs=['channels'],
    sensor_scales=4.5,
    sensor_colors={'fnirs': fnirs_colors},
)
brain.plotter.add_point_labels(
    mri_positions_mm + label_offset,
    labels,
    font_size=22,
    text_color="black",
    point_color="white",
    point_size=1,
    show_points=False,
    always_visible=True,
    shape="rounded_rect",
    shape_color="white",
    shape_opacity=0.85,
    margin=2,
    name="ch_numbers",
)

# Adjust focal point to centre on the prefrontal sensor cluster.
sensors_centre = mri_positions_mm.mean(axis=0)
print("Sensors centre (MNI mm):", sensors_centre)

variants = [
    ('dorsal_default', dict(view='dorsal')),
    ('dorsal_tilt20',  dict(view='dorsal', elevation=70)),
    ('dorsal_close',   dict(view='dorsal', focalpoint=tuple(sensors_centre), distance=210)),
    ('dorsal_close_tilt', dict(view='dorsal', elevation=75, focalpoint=tuple(sensors_centre), distance=240)),
]
for name, kwargs in variants:
    brain.show_view(**kwargs)
    p = OUT_DIR / f'brain_v2_{name}.png'
    brain.save_image(str(p), mode='rgba')
    print(f"  {name:24s} → {p.name}")

Sensors centre (MNI mm): [ 0.96561965 50.38832576 41.7482759 ]


  dorsal_default           → brain_v2_dorsal_default.png


  dorsal_tilt20            → brain_v2_dorsal_tilt20.png
  dorsal_close             → brain_v2_dorsal_close.png


  dorsal_close_tilt        → brain_v2_dorsal_close_tilt.png


In [17]:
# Final brain figure: view='dorsal' + tight crop around sensors.
brain.show_view(view='dorsal')

# Use a temporary 'full' file, then crop tightly to the sensor cluster.
fig_brain_path = REPO / 'research' / 'paper-materials' / 'figures' / 'fig_montage_brain.png'
brain.save_image(str(fig_brain_path), mode='rgba')
img = iio.imread(str(fig_brain_path))
print("Full image:", img.shape)

# Tight crop around the dots (find dot bounding box from colored content).
# Background is near-white; dots and labels are coloured.
rgb = img[:, :, :3].astype(np.int32)
is_dot = (rgb.sum(axis=2) < 600)  # exclude brain (which is medium grey) — focus on the saturated dots
ys, xs = np.where(is_dot)
print("Dot bbox pixels:", len(ys))
if len(ys) == 0:
    print("ERROR: no dots found; widening threshold")
else:
    y0, y1 = ys.min(), ys.max()
    x0, x1 = xs.min(), xs.max()
    # Add generous padding so brain context is preserved.
    pad_top, pad_bot = 50, 200       # more below to keep frontal-lobe outline visible
    pad_lr = 80
    y0 = max(0, y0 - pad_top); y1 = min(img.shape[0], y1 + pad_bot)
    x0 = max(0, x0 - pad_lr);  x1 = min(img.shape[1], x1 + pad_lr)
    cropped = img[y0:y1, x0:x1]
    print("Cropped:", cropped.shape)
    iio.imwrite(str(fig_brain_path), cropped)

    # Re-write SVG wrapper.
    import base64
    svg_path = fig_brain_path.with_suffix(".svg")
    with open(fig_brain_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("ascii")
    H, W = cropped.shape[:2]
    svg = (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" '
        f'viewBox="0 0 {W} {H}">'
        f'<image href="data:image/png;base64,{b64}" width="{W}" height="{H}"/>'
        f'</svg>'
    )
    svg_path.write_text(svg)
    print("Saved cropped:", fig_brain_path.name, "+", svg_path.name)

Full image: (1200, 1400, 4)
Dot bbox pixels: 752579
Cropped: (1200, 1065, 4)
Saved cropped: fig_montage_brain.png + fig_montage_brain.svg


In [18]:
# Smarter crop: find COLOURED dots (high saturation), not the grey brain.
brain.show_view(view='dorsal')
brain.save_image(str(fig_brain_path), mode='rgba')
img = iio.imread(str(fig_brain_path))

rgb = img[:, :, :3].astype(np.int32)
# Saturation proxy: (max-min) channel difference. Coloured dots have big diff,
# grey brain has near-zero diff.
sat = rgb.max(axis=2) - rgb.min(axis=2)
is_dot = sat > 80
ys, xs = np.where(is_dot)
print("Saturated dot pixels:", len(ys))

y0, y1 = ys.min(), ys.max()
x0, x1 = xs.min(), xs.max()
print(f"Dot bbox: y=[{y0},{y1}], x=[{x0},{x1}]")

# Generous padding: ~half the sensor cluster's height below to keep frontal
# lobes visible.
sensor_h = y1 - y0
pad_top = 30
pad_bot = max(sensor_h * 2, 200)  # show ~2x sensor-height of brain below
pad_lr  = 60
y0 = max(0, y0 - pad_top); y1 = min(img.shape[0], y1 + pad_bot)
x0 = max(0, x0 - pad_lr);  x1 = min(img.shape[1], x1 + pad_lr)
cropped = img[y0:y1, x0:x1]
print("Cropped:", cropped.shape)
iio.imwrite(str(fig_brain_path), cropped)

import base64
svg_path = fig_brain_path.with_suffix(".svg")
with open(fig_brain_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("ascii")
H, W = cropped.shape[:2]
svg = (
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" '
    f'viewBox="0 0 {W} {H}">'
    f'<image href="data:image/png;base64,{b64}" width="{W}" height="{H}"/>'
    f'</svg>'
)
svg_path.write_text(svg)
print("Saved:", fig_brain_path)

Saturated dot pixels: 13656
Dot bbox: y=[5,297], x=[296,1104]
Cropped: (881, 928, 4)
Saved: /home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method/research/paper-materials/figures/fig_montage_brain.png


In [19]:
# Clean up intermediate preview files; keep only the final cropped Panel B PNG.
for p in OUT_DIR.glob('brain_preview_*.png'):
    p.unlink()
for p in OUT_DIR.glob('brain_v2_*.png'):
    p.unlink()
# Keep panel_A.{png,svg} and panel_B_brain.png in _intermediate for reproducibility.
print("Intermediate files now:")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size//1024} KB)")

Intermediate files now:
  panel_A.png  (170 KB)
  panel_A.svg  (47 KB)
  panel_B_brain.png  (400 KB)


In [20]:
# Revert to true frontal/rostral view (no elevation override) — matches the
# Image #3 reference: brain face-on with cerebellum at base, sensors arched
# across the forehead, both hemispheres visible side-by-side.
brain.close()

brain = mne.viz.Brain(
    "fsaverage", subjects_dir=str(subjects_dir), background="white",
    cortex="low_contrast", alpha=0.3, units='mm', size=(1400, 1200),
)
brain.add_sensors(
    info, trans="fsaverage", fnirs=['channels'],
    sensor_scales=4.5,
    sensor_colors={'fnirs': fnirs_colors},
)
brain.plotter.add_point_labels(
    mri_positions_mm + label_offset,
    labels,
    font_size=22,
    text_color="black",
    point_color="white",
    point_size=1,
    show_points=False,
    always_visible=True,
    shape="rounded_rect",
    shape_color="white",
    shape_opacity=0.85,
    margin=2,
    name="ch_numbers",
)

brain.show_view(view='rostral')   # NO elevation override — true frontal view

fig_brain_path = REPO / 'research' / 'paper-materials' / 'figures' / 'fig_montage_brain.png'
brain.save_image(str(fig_brain_path), mode='rgba')
img = iio.imread(str(fig_brain_path))
print("Saved (uncropped):", img.shape)

Saved (uncropped): (1200, 1400, 4)


In [21]:
# Crop tightly around the brain content + sensors (similar logic as before).
import base64
img = iio.imread(str(fig_brain_path))
rgb = img[:, :, :3].astype(np.int32)
sat = rgb.max(axis=2) - rgb.min(axis=2)
# Coloured dots OR brain (which is medium grey).
is_content = (sat > 30) | (rgb.sum(axis=2) < 600)
ys, xs = np.where(is_content)
y0, y1 = ys.min(), ys.max()
x0, x1 = xs.min(), xs.max()
pad = 50
y0 = max(0, y0 - pad); y1 = min(img.shape[0], y1 + pad)
x0 = max(0, x0 - pad); x1 = min(img.shape[1], x1 + pad)
cropped = img[y0:y1, x0:x1]
print("Cropped:", cropped.shape)
iio.imwrite(str(fig_brain_path), cropped)

svg_path = fig_brain_path.with_suffix(".svg")
with open(fig_brain_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("ascii")
H, W = cropped.shape[:2]
svg = (
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" '
    f'viewBox="0 0 {W} {H}">'
    f'<image href="data:image/png;base64,{b64}" width="{W}" height="{H}"/>'
    f'</svg>'
)
svg_path.write_text(svg)
print("Saved cropped PNG + SVG:", fig_brain_path.name)

Cropped: (1017, 920, 4)
Saved cropped PNG + SVG: fig_montage_brain.png


In [22]:
# Brain fix: shift channel-number labels DOWNWARD (negative z) so they sit
# on the cortex rather than floating above it.
brain.close()
brain = mne.viz.Brain(
    "fsaverage", subjects_dir=str(subjects_dir), background="white",
    cortex="low_contrast", alpha=0.3, units='mm', size=(1400, 1200),
)
brain.add_sensors(
    info, trans="fsaverage", fnirs=['channels'],
    sensor_scales=4.5,
    sensor_colors={'fnirs': fnirs_colors},
)

# NEW: negative z offset to shift labels DOWN into the brain area.
new_offset = np.zeros_like(mri_positions_mm)
new_offset[:, 2] = -8.0   # was +6, now -8 (shift label DOWN/INFERIOR)

brain.plotter.add_point_labels(
    mri_positions_mm + new_offset,
    labels,
    font_size=22,
    text_color="black",
    point_color="white",
    point_size=1,
    show_points=False,
    always_visible=True,
    shape="rounded_rect",
    shape_color="white",
    shape_opacity=0.85,
    margin=2,
    name="ch_numbers",
)
brain.show_view(view='rostral')

fig_brain_path = REPO / 'research' / 'paper-materials' / 'figures' / 'fig_montage_brain.png'
brain.save_image(str(fig_brain_path), mode='rgba')
img = iio.imread(str(fig_brain_path))
print("Saved (uncropped):", img.shape)

Saved (uncropped): (1200, 1400, 4)


In [23]:
# Re-crop the updated brain image.
import base64
img = iio.imread(str(fig_brain_path))
rgb = img[:, :, :3].astype(np.int32)
sat = rgb.max(axis=2) - rgb.min(axis=2)
is_content = (sat > 30) | (rgb.sum(axis=2) < 600)
ys, xs = np.where(is_content)
y0, y1 = ys.min(), ys.max()
x0, x1 = xs.min(), xs.max()
pad = 50
y0 = max(0, y0 - pad); y1 = min(img.shape[0], y1 + pad)
x0 = max(0, x0 - pad); x1 = min(img.shape[1], x1 + pad)
cropped = img[y0:y1, x0:x1]
print("Cropped:", cropped.shape)
iio.imwrite(str(fig_brain_path), cropped)

svg_path = fig_brain_path.with_suffix(".svg")
with open(fig_brain_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("ascii")
H, W = cropped.shape[:2]
svg = (
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" '
    f'viewBox="0 0 {W} {H}">'
    f'<image href="data:image/png;base64,{b64}" width="{W}" height="{H}"/></svg>'
)
svg_path.write_text(svg)
print("Done.")

Cropped: (971, 920, 4)
Done.


In [24]:
# Brain: increase label inferior shift from -8 mm to -15 mm so labels clearly
# sit on the cortex surface rather than near the top edge.
brain.close()
brain = mne.viz.Brain(
    "fsaverage", subjects_dir=str(subjects_dir), background="white",
    cortex="low_contrast", alpha=0.3, units='mm', size=(1400, 1200),
)
brain.add_sensors(
    info, trans="fsaverage", fnirs=['channels'],
    sensor_scales=4.5,
    sensor_colors={'fnirs': fnirs_colors},
)
new_offset = np.zeros_like(mri_positions_mm)
new_offset[:, 2] = -15.0   # push label well inside the cortex
brain.plotter.add_point_labels(
    mri_positions_mm + new_offset,
    labels,
    font_size=22,
    text_color="black",
    point_color="white",
    point_size=1,
    show_points=False,
    always_visible=True,
    shape="rounded_rect",
    shape_color="white",
    shape_opacity=0.85,
    margin=2,
    name="ch_numbers",
)
brain.show_view(view='rostral')
brain.save_image(str(fig_brain_path), mode='rgba')

# Re-crop.
img = iio.imread(str(fig_brain_path))
rgb = img[:, :, :3].astype(np.int32)
sat = rgb.max(axis=2) - rgb.min(axis=2)
is_content = (sat > 30) | (rgb.sum(axis=2) < 600)
ys, xs = np.where(is_content)
y0, y1 = ys.min(), ys.max()
x0, x1 = xs.min(), xs.max()
pad = 50
y0 = max(0, y0 - pad); y1 = min(img.shape[0], y1 + pad)
x0 = max(0, x0 - pad); x1 = min(img.shape[1], x1 + pad)
cropped = img[y0:y1, x0:x1]
iio.imwrite(str(fig_brain_path), cropped)
print("Saved cropped brain:", cropped.shape)

import base64
svg_path = fig_brain_path.with_suffix(".svg")
with open(fig_brain_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("ascii")
H, W = cropped.shape[:2]
svg = (
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" '
    f'viewBox="0 0 {W} {H}">'
    f'<image href="data:image/png;base64,{b64}" width="{W}" height="{H}"/></svg>'
)
svg_path.write_text(svg)
print("Done.")

Saved cropped brain: (971, 920, 4)
Done.


In [25]:
# Project sensor 3D positions INWARD toward the brain centre so the dots
# land on the cortex (not floating above the scalp).
brain.close()

# Compute brain centroid from the pial surface.
import mne
surf_dir = subjects_dir / "fsaverage" / "surf"
verts_lh, _ = mne.read_surface(str(surf_dir / "lh.pial"))
verts_rh, _ = mne.read_surface(str(surf_dir / "rh.pial"))
brain_centroid_mm = np.concatenate([verts_lh, verts_rh], axis=0).mean(axis=0)
print("Brain centroid (MNI mm):", brain_centroid_mm)

# For each sensor, shift INWARD along the line from the sensor to the centroid.
# 15 mm should bring scalp-surface sensors onto the cortex.
direction = brain_centroid_mm - mri_positions_mm
direction_norm = direction / np.linalg.norm(direction, axis=1, keepdims=True)
shift_inward_mm = 16.0
mri_positions_on_cortex = mri_positions_mm + direction_norm * shift_inward_mm
print("Sample (S1_D1): scalp", mri_positions_mm[0], "→ cortex", mri_positions_on_cortex[0])

# Build a synthetic Info where the channel positions are now on the cortex.
# Easiest: take the existing info and rewrite each ch's loc[0:3] to the
# inward-shifted position. Note: loc is in HEAD coords (metres) — we have
# MRI mm — apply the inverse trans.
inv_trans = np.linalg.inv(trans["trans"])
head_positions_new_m = mne.transforms.apply_trans(inv_trans,
                                                    mri_positions_on_cortex / 1000.0)
info_proj = info.copy()
for i, ch in enumerate(info_proj["chs"]):
    new_loc = ch["loc"].copy()
    new_loc[:3] = head_positions_new_m[i]
    ch["loc"] = new_loc
print("Updated info_proj first chan loc[:3] (head m):", info_proj["chs"][0]["loc"][:3])

Brain centroid (MNI mm): [  0.24541178 -21.33126909  17.81609676]
Sample (S1_D1): scalp [-18.53291117  69.45304574   6.67208229] → cortex [-15.3151541   53.89671136   8.58166343]
Updated info_proj first chan loc[:3] (head m): [-0.01688462  0.08544469  0.04511318]


In [26]:
# Re-render the brain with the inward-projected sensor positions.
brain = mne.viz.Brain(
    "fsaverage", subjects_dir=str(subjects_dir), background="white",
    cortex="low_contrast", alpha=0.3, units='mm', size=(1400, 1200),
)
brain.add_sensors(
    info_proj, trans="fsaverage", fnirs=['channels'],
    sensor_scales=4.5,
    sensor_colors={'fnirs': fnirs_colors},
)
# Label positions also use the projected (on-cortex) coords, with a small
# inferior shift so labels sit just below each dot.
label_offset_v2 = np.zeros_like(mri_positions_on_cortex)
label_offset_v2[:, 2] = -4.0
brain.plotter.add_point_labels(
    mri_positions_on_cortex + label_offset_v2,
    labels,
    font_size=22,
    text_color="black",
    point_color="white",
    point_size=1,
    show_points=False,
    always_visible=True,
    shape="rounded_rect",
    shape_color="white",
    shape_opacity=0.85,
    margin=2,
    name="ch_numbers",
)
brain.show_view(view='rostral')

brain.save_image(str(fig_brain_path), mode='rgba')
print("Saved:", iio.imread(str(fig_brain_path)).shape)

Saved: (1200, 1400, 4)


In [27]:
# Crop tightly + write SVG.
import base64
img = iio.imread(str(fig_brain_path))
rgb = img[:, :, :3].astype(np.int32)
sat = rgb.max(axis=2) - rgb.min(axis=2)
is_content = (sat > 30) | (rgb.sum(axis=2) < 600)
ys, xs = np.where(is_content)
y0, y1 = ys.min(), ys.max()
x0, x1 = xs.min(), xs.max()
pad = 50
y0 = max(0, y0 - pad); y1 = min(img.shape[0], y1 + pad)
x0 = max(0, x0 - pad); x1 = min(img.shape[1], x1 + pad)
cropped = img[y0:y1, x0:x1]
iio.imwrite(str(fig_brain_path), cropped)
print("Cropped:", cropped.shape)

svg_path = fig_brain_path.with_suffix(".svg")
with open(fig_brain_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("ascii")
H, W = cropped.shape[:2]
svg = (
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" '
    f'viewBox="0 0 {W} {H}">'
    f'<image href="data:image/png;base64,{b64}" width="{W}" height="{H}"/></svg>'
)
svg_path.write_text(svg)

Cropped: (889, 935, 4)


570156

In [28]:
# Brain: keep cortex projection AND additionally shift dots -10 mm in z
# so they move visibly downward in the rostral (front) view.
brain.close()

# Same cortex projection (16 mm toward centroid) then -10 mm z (down in image).
mri_positions_v3 = mri_positions_on_cortex.copy()
mri_positions_v3[:, 2] -= 10.0   # shift inferior by 10 mm

# Re-derive head-coord positions from updated MRI positions and patch info.
inv_trans = np.linalg.inv(trans["trans"])
head_positions_v3_m = mne.transforms.apply_trans(inv_trans, mri_positions_v3 / 1000.0)
info_v3 = info.copy()
for i, ch in enumerate(info_v3["chs"]):
    new_loc = ch["loc"].copy()
    new_loc[:3] = head_positions_v3_m[i]
    ch["loc"] = new_loc

brain = mne.viz.Brain(
    "fsaverage", subjects_dir=str(subjects_dir), background="white",
    cortex="low_contrast", alpha=0.3, units='mm', size=(1400, 1200),
)
brain.add_sensors(
    info_v3, trans="fsaverage", fnirs=['channels'],
    sensor_scales=4.5,
    sensor_colors={'fnirs': fnirs_colors},
)
label_offset_v3 = np.zeros_like(mri_positions_v3)
label_offset_v3[:, 2] = -4.0
brain.plotter.add_point_labels(
    mri_positions_v3 + label_offset_v3,
    labels,
    font_size=22,
    text_color="black",
    point_color="white",
    point_size=1,
    show_points=False,
    always_visible=True,
    shape="rounded_rect",
    shape_color="white",
    shape_opacity=0.85,
    margin=2,
    name="ch_numbers",
)
brain.show_view(view='rostral')
brain.save_image(str(fig_brain_path), mode='rgba')
print("Saved.")

# Crop + SVG
import base64
img = iio.imread(str(fig_brain_path))
rgb = img[:, :, :3].astype(np.int32)
sat = rgb.max(axis=2) - rgb.min(axis=2)
is_content = (sat > 30) | (rgb.sum(axis=2) < 600)
ys, xs = np.where(is_content)
y0, y1 = ys.min(), ys.max(); x0, x1 = xs.min(), xs.max()
pad = 50
y0 = max(0, y0-pad); y1 = min(img.shape[0], y1+pad)
x0 = max(0, x0-pad); x1 = min(img.shape[1], x1+pad)
cropped = img[y0:y1, x0:x1]
iio.imwrite(str(fig_brain_path), cropped)

svg_path = fig_brain_path.with_suffix(".svg")
with open(fig_brain_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("ascii")
H, W = cropped.shape[:2]
svg_path.write_text(
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" '
    f'viewBox="0 0 {W} {H}">'
    f'<image href="data:image/png;base64,{b64}" width="{W}" height="{H}"/></svg>'
)
print("Cropped:", cropped.shape)

Saved.
Cropped: (871, 935, 4)


In [29]:
# Brain v4: smaller inward projection (8 mm), no extra z shift.
# Keeps dots near the cortex surface from outside, so they stay visible in
# rostral view (no occlusion by the brain volume).
brain.close()

shift_inward_v4 = 8.0
direction = brain_centroid_mm - mri_positions_mm
direction_norm = direction / np.linalg.norm(direction, axis=1, keepdims=True)
mri_positions_v4 = mri_positions_mm + direction_norm * shift_inward_v4
# No additional z shift this time.

# Patch info loc with new positions.
inv_trans = np.linalg.inv(trans["trans"])
head_positions_v4_m = mne.transforms.apply_trans(inv_trans, mri_positions_v4 / 1000.0)
info_v4 = info.copy()
for i, ch in enumerate(info_v4["chs"]):
    new_loc = ch["loc"].copy()
    new_loc[:3] = head_positions_v4_m[i]
    ch["loc"] = new_loc

brain = mne.viz.Brain(
    "fsaverage", subjects_dir=str(subjects_dir), background="white",
    cortex="low_contrast", alpha=0.3, units='mm', size=(1400, 1200),
)
brain.add_sensors(
    info_v4, trans="fsaverage", fnirs=['channels'],
    sensor_scales=4.5,
    sensor_colors={'fnirs': fnirs_colors},
)
label_offset_v4 = np.zeros_like(mri_positions_v4)
label_offset_v4[:, 2] = -4.0
brain.plotter.add_point_labels(
    mri_positions_v4 + label_offset_v4,
    labels,
    font_size=22,
    text_color="black",
    point_color="white",
    point_size=1,
    show_points=False,
    always_visible=True,
    shape="rounded_rect",
    shape_color="white",
    shape_opacity=0.85,
    margin=2,
    name="ch_numbers",
)
brain.show_view(view='rostral')
brain.save_image(str(fig_brain_path), mode='rgba')

# Crop + SVG
import base64
img = iio.imread(str(fig_brain_path))
rgb = img[:, :, :3].astype(np.int32)
sat = rgb.max(axis=2) - rgb.min(axis=2)
is_content = (sat > 30) | (rgb.sum(axis=2) < 600)
ys, xs = np.where(is_content)
y0, y1 = ys.min(), ys.max(); x0, x1 = xs.min(), xs.max()
pad = 50
y0 = max(0, y0-pad); y1 = min(img.shape[0], y1+pad)
x0 = max(0, x0-pad); x1 = min(img.shape[1], x1+pad)
cropped = img[y0:y1, x0:x1]
iio.imwrite(str(fig_brain_path), cropped)
svg_path = fig_brain_path.with_suffix(".svg")
with open(fig_brain_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("ascii")
H, W = cropped.shape[:2]
svg_path.write_text(
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" '
    f'viewBox="0 0 {W} {H}">'
    f'<image href="data:image/png;base64,{b64}" width="{W}" height="{H}"/></svg>'
)
print("Cropped:", cropped.shape)

Cropped: (939, 935, 4)


In [30]:
# Brain v5: inward projection 12 mm (was 8), no z shift — push dots deeper
# into the cortex while keeping them visible.
brain.close()

shift_inward_v5 = 12.0
direction = brain_centroid_mm - mri_positions_mm
direction_norm = direction / np.linalg.norm(direction, axis=1, keepdims=True)
mri_positions_v5 = mri_positions_mm + direction_norm * shift_inward_v5

inv_trans = np.linalg.inv(trans["trans"])
head_positions_v5_m = mne.transforms.apply_trans(inv_trans, mri_positions_v5 / 1000.0)
info_v5 = info.copy()
for i, ch in enumerate(info_v5["chs"]):
    new_loc = ch["loc"].copy()
    new_loc[:3] = head_positions_v5_m[i]
    ch["loc"] = new_loc

brain = mne.viz.Brain(
    "fsaverage", subjects_dir=str(subjects_dir), background="white",
    cortex="low_contrast", alpha=0.3, units='mm', size=(1400, 1200),
)
brain.add_sensors(
    info_v5, trans="fsaverage", fnirs=['channels'],
    sensor_scales=4.5,
    sensor_colors={'fnirs': fnirs_colors},
)
label_offset_v5 = np.zeros_like(mri_positions_v5)
label_offset_v5[:, 2] = -4.0
brain.plotter.add_point_labels(
    mri_positions_v5 + label_offset_v5,
    labels,
    font_size=22,
    text_color="black",
    point_color="white",
    point_size=1,
    show_points=False,
    always_visible=True,
    shape="rounded_rect",
    shape_color="white",
    shape_opacity=0.85,
    margin=2,
    name="ch_numbers",
)
brain.show_view(view='rostral')
brain.save_image(str(fig_brain_path), mode='rgba')

# Crop + SVG
import base64
img = iio.imread(str(fig_brain_path))
rgb = img[:, :, :3].astype(np.int32)
sat = rgb.max(axis=2) - rgb.min(axis=2)
is_content = (sat > 30) | (rgb.sum(axis=2) < 600)
ys, xs = np.where(is_content)
y0, y1 = ys.min(), ys.max(); x0, x1 = xs.min(), xs.max()
pad = 50
y0 = max(0, y0-pad); y1 = min(img.shape[0], y1+pad)
x0 = max(0, x0-pad); x1 = min(img.shape[1], x1+pad)
cropped = img[y0:y1, x0:x1]
iio.imwrite(str(fig_brain_path), cropped)

svg_path = fig_brain_path.with_suffix(".svg")
with open(fig_brain_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("ascii")
H, W = cropped.shape[:2]
svg_path.write_text(
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" '
    f'viewBox="0 0 {W} {H}">'
    f'<image href="data:image/png;base64,{b64}" width="{W}" height="{H}"/></svg>'
)
print("Cropped:", cropped.shape)

Cropped: (914, 935, 4)


In [31]:
# Brain v6: aggressive inward projection (18 mm) — push dots clearly inside
# the cortex. The cortex alpha=0.3 makes it translucent so dots remain visible
# through the cortex surface.
brain.close()

shift_inward_v6 = 18.0
direction = brain_centroid_mm - mri_positions_mm
direction_norm = direction / np.linalg.norm(direction, axis=1, keepdims=True)
mri_positions_v6 = mri_positions_mm + direction_norm * shift_inward_v6

inv_trans = np.linalg.inv(trans["trans"])
head_positions_v6_m = mne.transforms.apply_trans(inv_trans, mri_positions_v6 / 1000.0)
info_v6 = info.copy()
for i, ch in enumerate(info_v6["chs"]):
    new_loc = ch["loc"].copy()
    new_loc[:3] = head_positions_v6_m[i]
    ch["loc"] = new_loc

brain = mne.viz.Brain(
    "fsaverage", subjects_dir=str(subjects_dir), background="white",
    cortex="low_contrast", alpha=0.3, units='mm', size=(1400, 1200),
)
brain.add_sensors(
    info_v6, trans="fsaverage", fnirs=['channels'],
    sensor_scales=4.5,
    sensor_colors={'fnirs': fnirs_colors},
)
label_offset_v6 = np.zeros_like(mri_positions_v6)
label_offset_v6[:, 2] = -4.0
brain.plotter.add_point_labels(
    mri_positions_v6 + label_offset_v6,
    labels,
    font_size=22,
    text_color="black",
    point_color="white",
    point_size=1,
    show_points=False,
    always_visible=True,
    shape="rounded_rect",
    shape_color="white",
    shape_opacity=0.85,
    margin=2,
    name="ch_numbers",
)
brain.show_view(view='rostral')
brain.save_image(str(fig_brain_path), mode='rgba')

# Crop + SVG
import base64
img = iio.imread(str(fig_brain_path))
rgb = img[:, :, :3].astype(np.int32)
sat = rgb.max(axis=2) - rgb.min(axis=2)
is_content = (sat > 30) | (rgb.sum(axis=2) < 600)
ys, xs = np.where(is_content)
y0, y1 = ys.min(), ys.max(); x0, x1 = xs.min(), xs.max()
pad = 50
y0 = max(0, y0-pad); y1 = min(img.shape[0], y1+pad)
x0 = max(0, x0-pad); x1 = min(img.shape[1], x1+pad)
cropped = img[y0:y1, x0:x1]
iio.imwrite(str(fig_brain_path), cropped)

svg_path = fig_brain_path.with_suffix(".svg")
with open(fig_brain_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("ascii")
H, W = cropped.shape[:2]
svg_path.write_text(
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" '
    f'viewBox="0 0 {W} {H}">'
    f'<image href="data:image/png;base64,{b64}" width="{W}" height="{H}"/></svg>'
)
print("Cropped:", cropped.shape)

Cropped: (877, 935, 4)


In [32]:
# Brain v7: use brain.add_foci(map_surface='pial') so each sensor snaps to
# the closest pial-cortex vertex — clean on-cortex placement, no occlusion,
# no manual centroid projection.
brain.close()

brain = mne.viz.Brain(
    "fsaverage", subjects_dir=str(subjects_dir), background="white",
    cortex="low_contrast", alpha=0.35, units='mm', size=(1400, 1200),
)

# Group sensors by ROI so we can colour them per region using add_foci.
roi_groups = {
    "VMPFC": (vmpfc_idx, "blue"),
    "DMPFC": (dmpfc_idx, "green"),
    "DLPFC": (dlpfc_idx, "red"),
    "Default": (
        [i for i in range(n_ch)
         if i not in set(vmpfc_idx + dmpfc_idx + dlpfc_idx)],
        (1.0, 0.647, 0.0),    # orange
    ),
}

# add_foci expects coords in MRI mm for fsaverage; we already have
# mri_positions_mm. map_surface='pial' snaps each focus to the nearest
# vertex on the pial surface.
foci_positions: list[tuple[int, np.ndarray]] = []
for roi, (idx_list, color) in roi_groups.items():
    if not idx_list:
        continue
    coords = mri_positions_mm[idx_list]
    brain.add_foci(coords, map_surface='pial',
                   color=color, scale_factor=0.6, alpha=1.0)
    for i, c in zip(idx_list, coords):
        foci_positions.append((i, c))
print(f"Added {len(foci_positions)} foci across {len(roi_groups)} ROIs")

ValueError: hemi must not be None when both hemispheres are displayed

In [33]:
# Manual pial-surface projection using cKDTree (equivalent to what add_foci
# does internally with map_surface='pial').
from scipy.spatial import cKDTree
tree_lh = cKDTree(verts_lh)
tree_rh = cKDTree(verts_rh)

mri_positions_on_pial = np.zeros_like(mri_positions_mm)
for i, pos in enumerate(mri_positions_mm):
    d_lh, idx_lh = tree_lh.query(pos)
    d_rh, idx_rh = tree_rh.query(pos)
    if d_lh < d_rh:
        mri_positions_on_pial[i] = verts_lh[idx_lh]
    else:
        mri_positions_on_pial[i] = verts_rh[idx_rh]

print("Distance from scalp to nearest pial vertex (mm):")
distances = np.linalg.norm(mri_positions_mm - mri_positions_on_pial, axis=1)
print(f"  min={distances.min():.1f}, max={distances.max():.1f}, mean={distances.mean():.1f}")
print("Sample (S1_D1): scalp", mri_positions_mm[0],
      "→ pial", mri_positions_on_pial[0])

Distance from scalp to nearest pial vertex (mm):
  min=3.8, max=21.4, mean=11.8
Sample (S1_D1): scalp [-18.53291117  69.45304574   6.67208229] → pial [-15.94285393  65.28969574   7.62465954]


In [34]:
# Re-render brain with dots at exact pial-surface positions.
brain.close()

# Patch info loc with pial-projected positions.
head_positions_pial_m = mne.transforms.apply_trans(inv_trans, mri_positions_on_pial / 1000.0)
info_pial = info.copy()
for i, ch in enumerate(info_pial["chs"]):
    new_loc = ch["loc"].copy()
    new_loc[:3] = head_positions_pial_m[i]
    ch["loc"] = new_loc

brain = mne.viz.Brain(
    "fsaverage", subjects_dir=str(subjects_dir), background="white",
    cortex="low_contrast", alpha=0.35, units='mm', size=(1400, 1200),
)
brain.add_sensors(
    info_pial, trans="fsaverage", fnirs=['channels'],
    sensor_scales=4.5,
    sensor_colors={'fnirs': fnirs_colors},
)
label_offset_pial = np.zeros_like(mri_positions_on_pial)
label_offset_pial[:, 2] = -4.0
brain.plotter.add_point_labels(
    mri_positions_on_pial + label_offset_pial,
    labels,
    font_size=22,
    text_color="black",
    point_color="white",
    point_size=1,
    show_points=False,
    always_visible=True,
    shape="rounded_rect",
    shape_color="white",
    shape_opacity=0.85,
    margin=2,
    name="ch_numbers",
)
brain.show_view(view='rostral')
brain.save_image(str(fig_brain_path), mode='rgba')

# Crop + SVG
import base64
img = iio.imread(str(fig_brain_path))
rgb = img[:, :, :3].astype(np.int32)
sat = rgb.max(axis=2) - rgb.min(axis=2)
is_content = (sat > 30) | (rgb.sum(axis=2) < 600)
ys, xs = np.where(is_content)
y0, y1 = ys.min(), ys.max(); x0, x1 = xs.min(), xs.max()
pad = 50
y0 = max(0, y0-pad); y1 = min(img.shape[0], y1+pad)
x0 = max(0, x0-pad); x1 = min(img.shape[1], x1+pad)
cropped = img[y0:y1, x0:x1]
iio.imwrite(str(fig_brain_path), cropped)

svg_path = fig_brain_path.with_suffix(".svg")
with open(fig_brain_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("ascii")
H, W = cropped.shape[:2]
svg_path.write_text(
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" '
    f'viewBox="0 0 {W} {H}">'
    f'<image href="data:image/png;base64,{b64}" width="{W}" height="{H}"/></svg>'
)
print("Cropped:", cropped.shape)

Cropped: (871, 935, 4)


In [35]:
# Brain v8 (REVERT): go back to the 18 mm centroid projection.
brain.close()

shift_inward_v8 = 18.0
direction = brain_centroid_mm - mri_positions_mm
direction_norm = direction / np.linalg.norm(direction, axis=1, keepdims=True)
mri_positions_v8 = mri_positions_mm + direction_norm * shift_inward_v8

inv_trans = np.linalg.inv(trans["trans"])
head_positions_v8_m = mne.transforms.apply_trans(inv_trans, mri_positions_v8 / 1000.0)
info_v8 = info.copy()
for i, ch in enumerate(info_v8["chs"]):
    new_loc = ch["loc"].copy()
    new_loc[:3] = head_positions_v8_m[i]
    ch["loc"] = new_loc

brain = mne.viz.Brain(
    "fsaverage", subjects_dir=str(subjects_dir), background="white",
    cortex="low_contrast", alpha=0.3, units='mm', size=(1400, 1200),
)
brain.add_sensors(
    info_v8, trans="fsaverage", fnirs=['channels'],
    sensor_scales=4.5,
    sensor_colors={'fnirs': fnirs_colors},
)
label_offset_v8 = np.zeros_like(mri_positions_v8)
label_offset_v8[:, 2] = -4.0
brain.plotter.add_point_labels(
    mri_positions_v8 + label_offset_v8,
    labels,
    font_size=22,
    text_color="black",
    point_color="white",
    point_size=1,
    show_points=False,
    always_visible=True,
    shape="rounded_rect",
    shape_color="white",
    shape_opacity=0.85,
    margin=2,
    name="ch_numbers",
)
brain.show_view(view='rostral')
brain.save_image(str(fig_brain_path), mode='rgba')

# Crop + SVG
import base64
img = iio.imread(str(fig_brain_path))
rgb = img[:, :, :3].astype(np.int32)
sat = rgb.max(axis=2) - rgb.min(axis=2)
is_content = (sat > 30) | (rgb.sum(axis=2) < 600)
ys, xs = np.where(is_content)
y0, y1 = ys.min(), ys.max(); x0, x1 = xs.min(), xs.max()
pad = 50
y0 = max(0, y0-pad); y1 = min(img.shape[0], y1+pad)
x0 = max(0, x0-pad); x1 = min(img.shape[1], x1+pad)
cropped = img[y0:y1, x0:x1]
iio.imwrite(str(fig_brain_path), cropped)

svg_path = fig_brain_path.with_suffix(".svg")
with open(fig_brain_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("ascii")
H, W = cropped.shape[:2]
svg_path.write_text(
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" '
    f'viewBox="0 0 {W} {H}">'
    f'<image href="data:image/png;base64,{b64}" width="{W}" height="{H}"/></svg>'
)
print("Cropped:", cropped.shape)

Cropped: (877, 935, 4)


In [36]:
# Brain v9: deeper projection (24 mm) + labels ABOVE dots + no white background.
brain.close()

shift_inward_v9 = 24.0
direction = brain_centroid_mm - mri_positions_mm
direction_norm = direction / np.linalg.norm(direction, axis=1, keepdims=True)
mri_positions_v9 = mri_positions_mm + direction_norm * shift_inward_v9

inv_trans = np.linalg.inv(trans["trans"])
head_positions_v9_m = mne.transforms.apply_trans(inv_trans, mri_positions_v9 / 1000.0)
info_v9 = info.copy()
for i, ch in enumerate(info_v9["chs"]):
    new_loc = ch["loc"].copy()
    new_loc[:3] = head_positions_v9_m[i]
    ch["loc"] = new_loc

brain = mne.viz.Brain(
    "fsaverage", subjects_dir=str(subjects_dir), background="white",
    cortex="low_contrast", alpha=0.3, units='mm', size=(1400, 1200),
)
brain.add_sensors(
    info_v9, trans="fsaverage", fnirs=['channels'],
    sensor_scales=4.5,
    sensor_colors={'fnirs': fnirs_colors},
)
# Label ABOVE dot (positive z), no shape background.
label_offset_v9 = np.zeros_like(mri_positions_v9)
label_offset_v9[:, 2] = +5.0
brain.plotter.add_point_labels(
    mri_positions_v9 + label_offset_v9,
    labels,
    font_size=22,
    text_color="black",
    point_color="white",
    point_size=1,
    show_points=False,
    always_visible=True,
    shape=None,           # NO background
    margin=0,
    name="ch_numbers",
)
brain.show_view(view='rostral')
brain.save_image(str(fig_brain_path), mode='rgba')

# Crop + SVG
import base64
img = iio.imread(str(fig_brain_path))
rgb = img[:, :, :3].astype(np.int32)
sat = rgb.max(axis=2) - rgb.min(axis=2)
is_content = (sat > 30) | (rgb.sum(axis=2) < 600)
ys, xs = np.where(is_content)
y0, y1 = ys.min(), ys.max(); x0, x1 = xs.min(), xs.max()
pad = 50
y0 = max(0, y0-pad); y1 = min(img.shape[0], y1+pad)
x0 = max(0, x0-pad); x1 = min(img.shape[1], x1+pad)
cropped = img[y0:y1, x0:x1]
iio.imwrite(str(fig_brain_path), cropped)

svg_path = fig_brain_path.with_suffix(".svg")
with open(fig_brain_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("ascii")
H, W = cropped.shape[:2]
svg_path.write_text(
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" '
    f'viewBox="0 0 {W} {H}">'
    f'<image href="data:image/png;base64,{b64}" width="{W}" height="{H}"/></svg>'
)
print("Cropped:", cropped.shape)

Cropped: (881, 935, 4)


In [37]:
# Brain v10 (notebook 2_ style): use original scalp positions (no projection)
# so dots preserve their natural spacing; add explicit -z shift to move the
# cluster down in the rostral view.
brain.close()

# Use ORIGINAL mri_positions_mm (already at scalp) + explicit -z shift.
mri_positions_v10 = mri_positions_mm.copy()
mri_positions_v10[:, 2] -= 15.0   # shift 15 mm down (inferior)

inv_trans = np.linalg.inv(trans["trans"])
head_positions_v10_m = mne.transforms.apply_trans(inv_trans, mri_positions_v10 / 1000.0)
info_v10 = info.copy()
for i, ch in enumerate(info_v10["chs"]):
    new_loc = ch["loc"].copy()
    new_loc[:3] = head_positions_v10_m[i]
    ch["loc"] = new_loc

brain = mne.viz.Brain(
    "fsaverage", subjects_dir=str(subjects_dir), background="white",
    cortex="low_contrast", alpha=0.3, units='mm', size=(1400, 1200),
)
brain.add_sensors(
    info_v10, trans="fsaverage", fnirs=['channels'],
    sensor_scales=4.5,
    sensor_colors={'fnirs': fnirs_colors},
)
label_offset_v10 = np.zeros_like(mri_positions_v10)
label_offset_v10[:, 2] = +5.0   # label ABOVE each dot
brain.plotter.add_point_labels(
    mri_positions_v10 + label_offset_v10,
    labels,
    font_size=22,
    text_color="black",
    point_color="white",
    point_size=1,
    show_points=False,
    always_visible=True,
    shape=None,
    margin=0,
    name="ch_numbers",
)
brain.show_view(view='rostral')
brain.save_image(str(fig_brain_path), mode='rgba')

# Crop + SVG
import base64
img = iio.imread(str(fig_brain_path))
rgb = img[:, :, :3].astype(np.int32)
sat = rgb.max(axis=2) - rgb.min(axis=2)
is_content = (sat > 30) | (rgb.sum(axis=2) < 600)
ys, xs = np.where(is_content)
y0, y1 = ys.min(), ys.max(); x0, x1 = xs.min(), xs.max()
pad = 50
y0 = max(0, y0-pad); y1 = min(img.shape[0], y1+pad)
x0 = max(0, x0-pad); x1 = min(img.shape[1], x1+pad)
cropped = img[y0:y1, x0:x1]
iio.imwrite(str(fig_brain_path), cropped)

svg_path = fig_brain_path.with_suffix(".svg")
with open(fig_brain_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("ascii")
H, W = cropped.shape[:2]
svg_path.write_text(
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" '
    f'viewBox="0 0 {W} {H}">'
    f'<image href="data:image/png;base64,{b64}" width="{W}" height="{H}"/></svg>'
)
print("Cropped:", cropped.shape)

Cropped: (908, 919, 4)


In [38]:
# Brain v11: contract cluster + more z-shift + label closer to dot.
brain.close()

# Step 1: start from notebook 2_ style scalp positions.
base = mri_positions_mm.copy()
# Step 2: contract the cluster ~15 % toward its own centroid.
cluster_centre = base.mean(axis=0)
shrink_factor = 0.85
base_shrunk = cluster_centre + (base - cluster_centre) * shrink_factor
# Step 3: explicit -20 mm z shift (more inferior than before).
mri_positions_v11 = base_shrunk.copy()
mri_positions_v11[:, 2] -= 20.0

inv_trans = np.linalg.inv(trans["trans"])
head_positions_v11_m = mne.transforms.apply_trans(inv_trans, mri_positions_v11 / 1000.0)
info_v11 = info.copy()
for i, ch in enumerate(info_v11["chs"]):
    new_loc = ch["loc"].copy()
    new_loc[:3] = head_positions_v11_m[i]
    ch["loc"] = new_loc

brain = mne.viz.Brain(
    "fsaverage", subjects_dir=str(subjects_dir), background="white",
    cortex="low_contrast", alpha=0.3, units='mm', size=(1400, 1200),
)
brain.add_sensors(
    info_v11, trans="fsaverage", fnirs=['channels'],
    sensor_scales=4.5,
    sensor_colors={'fnirs': fnirs_colors},
)
# Label offset reduced from +5 to +2 (closer to dot).
label_offset_v11 = np.zeros_like(mri_positions_v11)
label_offset_v11[:, 2] = +2.0
brain.plotter.add_point_labels(
    mri_positions_v11 + label_offset_v11,
    labels,
    font_size=22,
    text_color="black",
    point_color="white",
    point_size=1,
    show_points=False,
    always_visible=True,
    shape=None,
    margin=0,
    name="ch_numbers",
)
brain.show_view(view='rostral')
brain.save_image(str(fig_brain_path), mode='rgba')

# Crop + SVG
import base64
img = iio.imread(str(fig_brain_path))
rgb = img[:, :, :3].astype(np.int32)
sat = rgb.max(axis=2) - rgb.min(axis=2)
is_content = (sat > 30) | (rgb.sum(axis=2) < 600)
ys, xs = np.where(is_content)
y0, y1 = ys.min(), ys.max(); x0, x1 = xs.min(), xs.max()
pad = 50
y0 = max(0, y0-pad); y1 = min(img.shape[0], y1+pad)
x0 = max(0, x0-pad); x1 = min(img.shape[1], x1+pad)
cropped = img[y0:y1, x0:x1]
iio.imwrite(str(fig_brain_path), cropped)

svg_path = fig_brain_path.with_suffix(".svg")
with open(fig_brain_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("ascii")
H, W = cropped.shape[:2]
svg_path.write_text(
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" '
    f'viewBox="0 0 {W} {H}">'
    f'<image href="data:image/png;base64,{b64}" width="{W}" height="{H}"/></svg>'
)
print("Cropped:", cropped.shape)

Cropped: (870, 933, 4)


In [39]:
# Clean up intermediate preview files; keep the final brain PNG cache.
intermediate_dir = REPO / 'research' / 'paper-materials' / 'figures' / '_intermediate'
for p in intermediate_dir.glob('panel_A.*'):
    p.unlink()
print("Kept intermediates:")
for p in sorted(intermediate_dir.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size//1024} KB)")

Kept intermediates:
  panel_B_brain.png  (400 KB)


In [40]:
# XAI brain-overlay figures — Zhang Fig 4C (continuous) + Fig 5 (top-5)
# Headline-XAI cell per memory: ST x HbO x LOSO x mt2 native
import pandas as pd
from scipy.spatial.distance import cdist

xai_csv = REPO / 'research/xai/st/hbo/loso/mt2/native/node_importance.csv'
df = pd.read_csv(xai_csv).set_index('channel')
# Align to CHANNEL_NAMES order (== info.ch_names order minus ' hbo')
CHANNEL_NAMES = ['S1_D1','S1_D3','S2_D2','S2_D1','S2_D5','S3_D1','S3_D3','S3_D4','S3_D6',
                 'S4_D4','S4_D5','S4_D7','S5_D2','S5_D5','S5_D8','S6_D3','S6_D6',
                 'S7_D4','S7_D6','S7_D7','S8_D5','S8_D7','S8_D8']
importance = df.loc[CHANNEL_NAMES, 'mean'].to_numpy()
ranks = df.loc[CHANNEL_NAMES, 'rank'].to_numpy()
print("importance range:", importance.min(), "→", importance.max())
print("top-5 channels by rank:")
top5_idx = np.argsort(ranks)[:5]
for i in top5_idx:
    print(f"  rank {int(ranks[i]):2d}: {CHANNEL_NAMES[i]:7s}  importance={importance[i]:.4f}")
print("Channel positions already in scope as mri_positions_mm (23 x 3, MRI mm).")
print("Pial vertex arrays already in scope: verts_lh", verts_lh.shape, "verts_rh", verts_rh.shape)

importance range: 0.9825753 → 1.01354396
top-5 channels by rank:
  rank  1: S5_D5    importance=1.0135
  rank  2: S7_D4    importance=1.0115
  rank  3: S4_D4    importance=1.0110
  rank  4: S8_D5    importance=1.0099
  rank  5: S1_D1    importance=1.0094
Channel positions already in scope as mri_positions_mm (23 x 3, MRI mm).
Pial vertex arrays already in scope: verts_lh (163842, 3) verts_rh (163842, 3)


In [41]:
# Compute Gaussian-weighted vertex values for ALL 23 and for TOP-5.
sigma_mm = 15.0

def compute_vertex_overlay(verts, ch_positions, ch_importance, sigma):
    # Gaussian-weighted sum across channels: value(v) = sum_c imp_c * exp(-d(v,c)^2 / 2*sigma^2)
    d2 = cdist(verts, ch_positions) ** 2          # (N_v, n_ch)
    weights = np.exp(-d2 / (2 * sigma * sigma))   # (N_v, n_ch)
    return weights @ ch_importance                # (N_v,)

# Center the importance values so the colormap shows true relative contrast.
# (GAT attention is normalized, near 1; we visualize deviations from cluster mean.)
imp_centered = importance - importance.mean()

# Variant A — ALL 23 channels (continuous Fig 4C style)
values_lh_all = compute_vertex_overlay(verts_lh, mri_positions_mm, imp_centered, sigma_mm)
values_rh_all = compute_vertex_overlay(verts_rh, mri_positions_mm, imp_centered, sigma_mm)
print("ALL-23 vertex map: lh range", values_lh_all.min(), "→", values_lh_all.max())

# Variant B — TOP-5 only (Fig 5 style)
top5_pos = mri_positions_mm[top5_idx]
top5_imp = imp_centered[top5_idx]
values_lh_top5 = compute_vertex_overlay(verts_lh, top5_pos, top5_imp, sigma_mm)
values_rh_top5 = compute_vertex_overlay(verts_rh, top5_pos, top5_imp, sigma_mm)
print("TOP-5 vertex map: lh range", values_lh_top5.min(), "→", values_lh_top5.max())

ALL-23 vertex map: lh range -0.01645609544724001 → 0.008738711993355412
TOP-5 vertex map: lh range 1.958654023596727e-30 → 0.011286649742631863


In [42]:
# Build a reusable function: render Brain with vertex overlay + 4 views.
out_intermediate = REPO / 'research/paper-materials/figures/_intermediate'
out_intermediate.mkdir(exist_ok=True, parents=True)

def render_overlay_4views(values_lh, values_rh, stem, cmap, threshold_frac=0.10):
    """Render 4-view brain with per-vertex Gaussian-overlay heatmap.
    Saves 4 PNG screenshots under _intermediate/{stem}_{view}.png.
    """
    brain.close() if 'brain' in globals() and brain is not None else None  # safety
    b = mne.viz.Brain(
        "fsaverage", subjects_dir=str(subjects_dir), background="white",
        cortex="low_contrast", alpha=0.95, units='mm', size=(1100, 900),
    )

    # Apply transparency below threshold_frac * max so only the active prefrontal
    # zone is coloured.
    vmax = max(abs(values_lh).max(), abs(values_rh).max())
    fmin = threshold_frac * vmax
    fmid = (fmin + vmax) / 2
    fmax = vmax

    b.add_data(values_lh, hemi='lh', colormap=cmap,
               smoothing_steps=8,
               fmin=fmin, fmid=fmid, fmax=fmax,
               colorbar=True, transparent=True)
    b.add_data(values_rh, hemi='rh', colormap=cmap,
               smoothing_steps=8,
               fmin=fmin, fmid=fmid, fmax=fmax,
               colorbar=False, transparent=True)

    view_specs = {
        'rostral':  dict(view='rostral'),
        'dorsal':   dict(view='dorsal'),
        'lat_lh':   dict(azimuth=180, elevation=90, hemi='lh'),  # left lateral
        'lat_rh':   dict(azimuth=0,   elevation=90, hemi='rh'),  # right lateral
    }
    paths = {}
    for name, kw in view_specs.items():
        try:
            b.show_view(**kw)
        except Exception as e:
            print(f"  {name}: show_view failed ({e}); falling back to defaults")
            b.show_view()
        p = out_intermediate / f"{stem}_{name}.png"
        b.save_image(str(p), mode='rgba')
        paths[name] = p
        print(f"  saved {p.name}")
    b.close()
    return paths

# Variant A — continuous, all 23 channels, RdYlBu_r (Zhang-style cool→warm)
paths_all = render_overlay_4views(values_lh_all, values_rh_all,
                                   stem='xai_overlay_all23',
                                   cmap='RdYlBu_r',
                                   threshold_frac=0.05)

  saved xai_overlay_all23_rostral.png
  saved xai_overlay_all23_dorsal.png


  saved xai_overlay_all23_lat_lh.png
  saved xai_overlay_all23_lat_rh.png


In [43]:
# Improved rendering: use RAW importance (not centered) so the heatmap reflects
# absolute attention magnitude; use inferno colormap (dark purple → bright yellow)
# matching Zhang Fig 4C style. Threshold at 30 % so only the active prefrontal
# region is coloured (rest stays grey cortex).

def render_overlay_4views_v2(values_lh, values_rh, stem, cmap='inferno',
                              threshold_pct=30):
    b = mne.viz.Brain(
        "fsaverage", subjects_dir=str(subjects_dir), background="white",
        cortex="low_contrast", alpha=0.95, units='mm', size=(1100, 900),
    )
    # Compute fmin/fmid/fmax from percentiles of the combined positive distribution.
    all_vals = np.concatenate([values_lh, values_rh])
    pos_vals = all_vals[all_vals > 0]
    fmin = np.percentile(pos_vals, threshold_pct)
    fmax = np.percentile(pos_vals, 99.5)
    fmid = (fmin + fmax) / 2

    b.add_data(values_lh, hemi='lh', colormap=cmap, smoothing_steps=10,
               fmin=fmin, fmid=fmid, fmax=fmax, colorbar=True, transparent=True)
    b.add_data(values_rh, hemi='rh', colormap=cmap, smoothing_steps=10,
               fmin=fmin, fmid=fmid, fmax=fmax, colorbar=False, transparent=True)

    view_specs = {
        'rostral':  dict(view='rostral'),
        'dorsal':   dict(view='dorsal'),
        'lat_lh':   dict(azimuth=180, elevation=90, hemi='lh'),
        'lat_rh':   dict(azimuth=0,   elevation=90, hemi='rh'),
    }
    paths = {}
    for name, kw in view_specs.items():
        try:
            b.show_view(**kw)
        except Exception as e:
            print(f"  {name}: {e}; using defaults")
            b.show_view()
        p = out_intermediate / f"{stem}_{name}.png"
        b.save_image(str(p), mode='rgba')
        paths[name] = p
        print(f"  saved {p.name}")
    b.close()
    return paths

# Compute vertex values from RAW importance (all positive, near 1.0).
values_lh_all_raw = compute_vertex_overlay(verts_lh, mri_positions_mm, importance, sigma_mm)
values_rh_all_raw = compute_vertex_overlay(verts_rh, mri_positions_mm, importance, sigma_mm)
values_lh_top5_raw = compute_vertex_overlay(verts_lh, top5_pos, importance[top5_idx], sigma_mm)
values_rh_top5_raw = compute_vertex_overlay(verts_rh, top5_pos, importance[top5_idx], sigma_mm)

print("ALL-23 (raw):", values_lh_all_raw.min(), "→", values_lh_all_raw.max())
print("TOP-5 (raw):",  values_lh_top5_raw.min(),  "→", values_lh_top5_raw.max())

paths_all = render_overlay_4views_v2(values_lh_all_raw, values_rh_all_raw,
                                      stem='xai_overlay_all23_v2',
                                      cmap='inferno', threshold_pct=85)

ALL-23 (raw): 2.8104833429571732e-22 → 2.718077054478772
TOP-5 (raw): 1.8221692888436783e-28 → 1.0175718580574347


  saved xai_overlay_all23_v2_rostral.png
  saved xai_overlay_all23_v2_dorsal.png


  saved xai_overlay_all23_v2_lat_lh.png
  saved xai_overlay_all23_v2_lat_rh.png


In [44]:
# Variant B — top-5 channels only (Zhang Fig 5 style)
paths_top5 = render_overlay_4views_v2(values_lh_top5_raw, values_rh_top5_raw,
                                       stem='xai_overlay_top5_v2',
                                       cmap='inferno', threshold_pct=85)

  saved xai_overlay_top5_v2_rostral.png
  saved xai_overlay_top5_v2_dorsal.png


  saved xai_overlay_top5_v2_lat_lh.png
  saved xai_overlay_top5_v2_lat_rh.png


In [45]:
# Composite the 4 views into a 2x2 panel for each variant.
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
plt.rcParams["font.family"] = ["DejaVu Sans", "sans-serif"]

def composite_4view_panel(paths, title, out_png_path):
    """4 view PNGs → 2x2 matplotlib panel with sub-labels."""
    view_order = [('rostral', 'A — Frontal'),
                  ('dorsal',  'B — Dorsal'),
                  ('lat_lh',  'C — Left lateral'),
                  ('lat_rh',  'D — Right lateral')]

    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    for ax, (name, sublabel) in zip(axes.flat, view_order):
        img = mpimg.imread(str(paths[name]))
        ax.imshow(img)
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.text(-0.02, 1.03, sublabel, transform=ax.transAxes,
                fontsize=14, fontweight='bold', va='top', ha='left')
    fig.suptitle(title, fontsize=15, fontweight='bold', y=0.995)
    plt.subplots_adjust(left=0.02, right=0.98, top=0.94, bottom=0.02,
                         wspace=0.04, hspace=0.08)
    fig.savefig(str(out_png_path), dpi=300, bbox_inches='tight', facecolor='white')
    svg_path = out_png_path.with_suffix('.svg')
    fig.savefig(str(svg_path), bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return out_png_path

fig_dir = REPO / 'research/paper-materials/figures'
final_all = composite_4view_panel(
    paths_all, "XAI channel-importance overlay — all 23 channels (HbO LOSO mt2)",
    fig_dir / 'fig_xai_brain_overlay_all23.png')
final_top5 = composite_4view_panel(
    paths_top5, "XAI channel-importance overlay — top-5 channels (HbO LOSO mt2)",
    fig_dir / 'fig_xai_brain_overlay_top5.png')
print("Final:", final_all.name, "+", final_top5.name)

Final: fig_xai_brain_overlay_all23.png + fig_xai_brain_overlay_top5.png
